In [1]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from utils import find_nearest_gene_atac_peaks, preprocessing_rna, preprocessing_atac, preprocessing_prot
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from tqdm import tqdm
tqdm.pandas()
from pybiomart import Server

Dataset has been downloaded from:
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM5008738

In [2]:
metadata = pd.read_csv("./data/GSE164378_sc.meta.data_3P.csv", index_col=0)
metadata

,nCount_ADT,nFeature_ADT,nCount_RNA,nFeature_RNA,orig.ident,lane,donor,time,celltype.l1,celltype.l2,celltype.l3,Phase,Batch
L1_AAACCCAAGAAACTCA,7535,217,10823,2915,SeuratProject,L1,P2,7,Mono,CD14 Mono,CD14 Mono,G1,Batch1
L1_AAACCCAAGACATACA,6013,209,5864,1617,SeuratProject,L1,P1,7,CD4 T,CD4 TCM,CD4 TCM_1,G1,Batch1
L1_AAACCCACAACTGGTT,6620,213,5067,1381,SeuratProject,L1,P4,2,CD8 T,CD8 Naive,CD8 Naive,S,Batch1
L1_AAACCCACACGTACTA,3567,202,4786,1890,SeuratProject,L1,P3,7,NK,NK,NK_2,G1,Batch1
L1_AAACCCACAGCATACT,6402,215,6505,1621,SeuratProject,L1,P4,7,CD8 T,CD8 Naive,CD8 Naive,G1,Batch1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
E2L8_TTTGTTGGTCGTGATT,4170,197,9346,2201,SeuratProject,E2L8,P5,7,CD8 T,CD8 Naive,CD8 Naive,S,Batch2
E2L8_TTTGTTGGTGTGCCTG,6927,209,9318,2938,SeuratProject,E2L8,P5,2,Mono,CD14 Mono,CD14 Mono,G1,Batch2
E2L8_TTTGTTGGTTAGTTCG,4222,206,11619,3224,SeuratProject,E2L8,P8,0,B,B intermediate,B intermediate kappa,S,Batch2
E2L8_TTTGTTGGTTGGCTAT,6063,217,15436,3999,SeuratProject,E2L8,P5,2,Mono,CD16 Mono,CD16 Mono,G1,Batch2


# RNA Layer Preprocessing

In [3]:
server = Server(host="http://www.ensembl.org")
dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["hsapiens_gene_ensembl"]
ensembl_gene_id_external_gene_name = dataset.query(attributes=["ensembl_gene_id", "external_gene_name"])
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name.rename(columns={"Gene stable ID": "gene_stable_id"})
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name[~ensembl_gene_id_external_gene_name["Gene name"].isnull()]
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name.groupby('Gene name')['gene_stable_id'].agg(lambda x: '{' + ', '.join(f'"{item}"' for item in set(x)) + '}' if len(x) > 1 else x.iloc[0]).reset_index()

In [4]:
rna = sc.read_mtx("./data/GSM5008737_RNA_3P-matrix.mtx.gz").T
rna_var = pd.read_csv("./data/GSM5008737_RNA_3P-features.tsv", header=None, delimiter="\t", index_col=0)
rna_var = rna_var.rename(columns={1: "gene_name", 2: "modality"})
rna_var.index.name = None
rna_var["gene_name"] = rna_var["gene_name"].apply(lambda x: x.split(".")[0])
rna_var = rna_var.join(ensembl_gene_id_external_gene_name.set_index("Gene name"), on="gene_name")
rna.var = rna_var
rna.obs = metadata
rna 

AnnData object with n_obs × n_vars = 161764 × 33538
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'Batch'
    var: 'gene_name', 'modality', 'gene_stable_id'

In [5]:
rna = preprocessing_rna(rna)
rna

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

AnnData object with n_obs × n_vars = 149058 × 25207
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'Batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_name', 'modality', 'gene_stable_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

# Prot Layer Preprocessing

In [6]:
prot = sc.read_mtx("./data/GSM5008738_ADT_3P-matrix.mtx.gz").T
prot_var = pd.read_csv("./data/GSM5008738_ADT_3P-features.tsv", header=None, delimiter="\t", index_col=0)
prot_var = prot_var.rename(columns={1: "gene_name", 2: "modality"})
prot_var.index.name = None
prot.var = prot_var
prot.obs = metadata
prot 

AnnData object with n_obs × n_vars = 161764 × 228
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'Batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_name', 'modality'

In [7]:
prot.var["gene_stable_id"] = ['ENSG00000138185',
np.nan,
'ENSG00000185896',
'ENSG00000174175',
'ENSG00000211772',
'ENSG00000120949',
'ENSG00000261371',
'ENSG00000174059',
'ENSG00000203710',
'ENSG00000135218',
'ENSG00000089692',
'ENSG00000181847',
'ENSG00000211716',
'ENSG00000150637',
'ENSG00000117560',
'ENSG00000026751',
'ENSG00000198910',
'ENSG00000105366',
'ENSG00000141736',
np.nan,
'ENSG00000037280',
'ENSG00000150093',
'ENSG00000007908',
'ENSG00000010610',
'ENSG00000010610',
'ENSG00000012124',
'ENSG00000261371',
'ENSG00000156738',
'ENSG00000139193',
'ENSG00000081237',
'ENSG00000134460',
'ENSG00000272398',
'ENSG00000076706',
'ENSG00000168961',
'ENSG00000117525',
'ENSG00000178726',
'ENSG00000183134',
np.nan,
'ENSG00000081237',
'ENSG00000168329',
'ENSG00000149294',
'ENSG00000149294',
'ENSG00000081237',
'ENSG00000198178',
'ENSG00000134352',
'ENSG00000121858',
'ENSG00000186891',
'ENSG00000169896',
'ENSG00000124731',
'ENSG00000169896',
'ENSG00000172322',
'ENSG00000004468',
'ENSG00000004468',
'ENSG00000110195',
np.nan,
'ENSG00000090659',
'ENSG00000163599',
'ENSG00000102245',
'ENSG00000073008',
'ENSG00000039068',
'ENSG00000101000',
'ENSG00000038945',
'ENSG00000054219',
'ENSG00000260314',
'ENSG00000116031',
'ENSG00000158473',
'ENSG00000136869',
'ENSG00000158481',
'ENSG00000162493',
'ENSG00000158477',
'ENSG00000135077',
'ENSG00000211898',
'ENSG00000211899',
'{"ENSG00000079385", "ENSG00000086548", "ENSG00000105388"}', 
'ENSG00000115232',
'ENSG00000173391',
'ENSG00000145850',
'ENSG00000168003',
'ENSG00000197992',
'ENSG00000213949',
'ENSG00000026508',
'ENSG00000134830',
'ENSG00000026508',
'ENSG00000167633',
'ENSG00000077238',
'ENSG00000168685',
'ENSG00000160712',
'ENSG00000188389',
'ENSG00000163600',
'ENSG00000185291',
'ENSG00000100385',
'ENSG00000153283',
'ENSG00000120217',
'ENSG00000026103',
'ENSG00000064300',
'ENSG00000157873',
'ENSG00000154096',
'ENSG00000186265',
'ENSG00000203747',
'ENSG00000170458',
'ENSG00000196371',
'ENSG00000166825',
'ENSG00000240505',
'ENSG00000197635',
'ENSG00000091972',
'ENSG00000197635',
'ENSG00000160255',
'ENSG00000177455',
'ENSG00000183813',
'ENSG00000211751',
'ENSG00000282506', 
'ENSG00000125726',
'ENSG00000072274',
'ENSG00000137101',
'ENSG00000135318',
'{"ENSG00000226660", "ENSG00000282568"}', 
'ENSG00000204936',
'ENSG00000132514',
'ENSG00000134853',
'ENSG00000113721',
'ENSG00000276053',
'ENSG00000099250',
'ENSG00000116824',
'ENSG00000128052',
'ENSG00000277092',
'ENSG00000117400',
'{"ENSG00000153563", "ENSG00000172116"}', 
'ENSG00000010278',
'ENSG00000204287',
'ENSG00000049249',
'ENSG00000186827',
'ENSG00000122025',
'ENSG00000259207',
'ENSG00000121807',
'ENSG00000159958',
'ENSG00000048462',
'ENSG00000110651',
'ENSG00000121594',
'ENSG00000112149',
'ENSG00000183625',
'ENSG00000205755',
'ENSG00000114013',
'ENSG00000184451',
'ENSG00000148400',
'ENSG00000134250',
'ENSG00000204475',
'ENSG00000007312',
'ENSG00000160223',
'ENSG00000160223',
'ENSG00000105369',
'ENSG00000164171',
'ENSG00000150337',
'ENSG00000135404',
'ENSG00000081237',
'ENSG00000081237',
'ENSG00000007062',
'ENSG00000007062',
'ENSG00000110848',
'ENSG00000129226',
'ENSG00000213809',
'{"ENSG00000005844", "ENSG00000160255"}', 
'ENSG00000172215',
'ENSG00000160683',
'ENSG00000121966',
'ENSG00000083457',
'ENSG00000108622',
'ENSG00000162692',
'ENSG00000106991',
'ENSG00000124469',
'ENSG00000117586',
'ENSG00000156535',
np.nan, 
'{"ENSG00000135424", "ENSG00000139626"}',
'ENSG00000172116',
'ENSG00000274143', 
'ENSG00000153563',
'ENSG00000154269',
'ENSG00000169442',
'ENSG00000160791',
'ENSG00000112486',
'ENSG00000109956',
'ENSG00000090339',
'ENSG00000196352',
'ENSG00000002586 ',
'ENSG00000085063',
'ENSG00000173585',
'ENSG00000125810',
'ENSG00000122223',
'ENSG00000163749',
'{"ENSG00000170180", "ENSG00000250361"}', 
'ENSG00000197646',
'ENSG00000085563',
'ENSG00000170558',
'ENSG00000039068',
'ENSG00000143297',
'ENSG00000198053',
'ENSG00000163518',
'ENSG00000185245',
'ENSG00000182578',
'ENSG00000157404',
'ENSG00000173578',
'ENSG00000130202',
'ENSG00000153208',
'ENSG00000134258',
'ENSG00000117322',
'ENSG00000160856',
'ENSG00000256594',
'ENSG00000117091',
'ENSG00000196776',
'ENSG00000117335',
'ENSG00000005961',
'ENSG00000101017',
'ENSG00000197471',
'ENSG00000118777',
'ENSG00000170180',
'ENSG00000189430',
'ENSG00000143226', 
'ENSG00000027697',
'ENSG00000088827',
'ENSG00000178562',
'ENSG00000111796',
'ENSG00000177575',
'ENSG00000115884',
'ENSG00000135535',
'ENSG00000115884',
'ENSG00000179776',
'ENSG00000120156',
'ENSG00000140678']

In [8]:
exclusion_features = ["Rat-IgG1-1", "Rat-IgG2b", "Rat-IgG1-2", "Rag-IgG2c"]

In [9]:
prot = prot[:,~prot.var["gene_name"].isin(exclusion_features)]
prot

View of AnnData object with n_obs × n_vars = 161764 × 224
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'Batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_name', 'modality', 'gene_stable_id'

In [10]:
prot = preprocessing_prot(prot)
prot

AnnData object with n_obs × n_vars = 161764 × 224
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'Batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_name', 'modality', 'gene_stable_id'
    uns: 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

# Create MuData and save

In [11]:
mdata = mu.MuData({"rna": rna, 
                   "prot": prot})

In [12]:
mu.pp.intersect_obs(mdata)

In [13]:
random_state = np.random.RandomState()
indices = random_state.permutation(list(range(len(mdata.obs))))
mdata = mdata[indices,:]

In [14]:
mdata.write("./data/Hao2021.h5mu")